# Fine-tuned vs Baseline — what is each model focused on?

Loads **both** models and runs them on the same sentence:
- **`my_bert_model2/`** — your fine-tuned model (real adverse-event head → classify + attention + masking).
- **`bert-base-uncased`** — off-the-shelf control (same architecture).

### Why the comparison is restricted to attention
Both encoders are identical in shape (12 heads, same positions `[3, 11]`), and attention comes from the **encoder**, not the head — so the *only* variable is fine-tuning. That makes attention a clean, fair comparison.

The baseline's classification head is **randomly initialised** (never trained), so:
- its `P(adverse)` is a meaningless random number, and
- masking it measures the drop of a random output, **not** a causal signal.

So the baseline is tagged **attention-only**, and its head outputs are shown with a warning, never as evidence. The headline you can defend: *fine-tuning reshapes what the encoder attends to.*

## 1 · Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


## 2 · Imports

In [4]:
import os, re
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, BertForSequenceClassification, BertConfig
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
print('imports ok')

imports ok


## 3 · Config

In [5]:
BASE_PATH     = "/content/drive/MyDrive/BERT_for_Disaster_Classification/Disaster_Tweets_Project"
FT_MODEL_PATH = f"{BASE_PATH}/my_bert_model2/"   # your fine-tuned model
BASELINE_NAME = "bert-base-uncased"              # off-the-shelf control (downloads from HF)

MODE = "fine"                      # "fine" or "ae_only"
ADVERSE_CLASS_INDEX = None         # auto-detect for fine-tuned; baseline head is random anyway

ATTENTION_LAYER   = -1
SPECIALIZED_HEADS = [3, 11]        # same positions in both models

ATTN_Z_THRESHOLD  = 1.0
ATTN_PCTL         = 75
DELTA_P_THRESHOLD = 0.02           # used only for the fine-tuned model
COMBINE_MODE      = "or"
ADVERSE_GATE      = 0.50           # used only for the fine-tuned model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## 4 · Load both models
Baseline gets a random head — we never trust its probability.

In [6]:
def _load(path):
    cfg = BertConfig.from_pretrained(path, output_attentions=True)
    m = BertForSequenceClassification.from_pretrained(path, config=cfg).to(device)
    tok = AutoTokenizer.from_pretrained(path)
    m.eval()
    return m, tok

ft_model, ft_tok = _load(FT_MODEL_PATH)
# baseline: num_labels=2 so the head exists (random) and shapes match; head is never trusted
bl_cfg = BertConfig.from_pretrained(BASELINE_NAME, output_attentions=True, num_labels=2)
bl_model = BertForSequenceClassification.from_pretrained(BASELINE_NAME, config=bl_cfg).to(device)
bl_tok = AutoTokenizer.from_pretrained(BASELINE_NAME)
bl_model.eval()

ADV_IDX = ADVERSE_CLASS_INDEX
if ADV_IDX is None:
    id2label = getattr(ft_model.config, "id2label", None) or {}
    ADV_IDX = 1
    for i, lab in id2label.items():
        if isinstance(lab, str) and re.search(r"adverse|positive|relevan|event|true", lab, re.I):
            ADV_IDX = int(i); break
print(f"fine-tuned adverse_class_index={ADV_IDX} | baseline head = RANDOM (not trusted)")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

fine-tuned adverse_class_index=1 | baseline head = RANDOM (not trusted)


## 5 · Tokenization (no spaCy)

In [7]:
WORD_RE = re.compile(r"\d+(?:\.\d+)?|[A-Za-z]+(?:'[A-Za-z]+)?|[^\w\s]")
def split_words(text): return WORD_RE.findall(text)

def encode_words(words, tok):
    enc = tok(words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=128)
    wid = enc.word_ids(0)
    return {k: v.to(device) for k, v in enc.items()}, wid

## 6 · Core signals (parameterised by model)

In [8]:
@torch.no_grad()
def classify(enc, model, adv_idx):
    out = model(**enc)
    return float(F.softmax(out.logits, dim=-1)[0][adv_idx]), out

@torch.no_grad()
def attention_saliency(out, word_ids, n_words):
    attn = out.attentions[ATTENTION_LAYER][0]
    n_heads = attn.shape[0]
    heads = [h for h in SPECIALIZED_HEADS if h < n_heads] or list(range(n_heads))
    cls_attn = attn[heads, 0, :].mean(dim=0)
    scores = np.zeros(n_words); counts = np.zeros(n_words)
    for s, w in enumerate(word_ids):
        if w is None: continue
        scores[w] += float(cls_attn[s]); counts[w] += 1
    counts[counts == 0] = 1
    scores /= counts
    return (scores - scores.mean()) / (scores.std() + 1e-9)

@torch.no_grad()
def masking_saliency(enc, word_ids, n_words, p_full, model, adv_idx, tok):
    mask_id = tok.mask_token_id; base = enc["input_ids"]; delta = np.zeros(n_words)
    w2p = {}
    for pos, w in enumerate(word_ids):
        if w is None: continue
        w2p.setdefault(w, []).append(pos)
    for w, pos in w2p.items():
        m = base.clone()
        for p in pos: m[0, p] = mask_id
        out = model(**{k: (m if k == "input_ids" else v) for k, v in enc.items()})
        delta[w] = p_full - float(F.softmax(out.logits, dim=-1)[0][adv_idx])
    return delta

## 7 · Saliency → BIO (shared rules)

In [9]:
def salient_words(attn_z, delta_p, use_masking):
    pctl = np.percentile(attn_z, ATTN_PCTL)
    attn_hit = (attn_z >= ATTN_Z_THRESHOLD) & (attn_z >= pctl)
    if not use_masking:
        return attn_hit                       # baseline: attention only
    mask_hit = (delta_p >= DELTA_P_THRESHOLD)
    return (attn_hit & mask_hit) if COMBINE_MODE == "and" else (attn_hit | mask_hit)

_WEEKDAYS={"monday","tuesday","wednesday","thursday","friday","saturday","sunday"}
_MONTHS={"january","february","march","april","may","june","july","august","september","october","november","december"}
_TIMEWORDS={"today","yesterday","tomorrow","tonight","morning","afternoon","evening","night","week","weekend","month","year"}
def is_time(t): return t.lower() in _WEEKDAYS or t.lower() in _MONTHS or t.lower() in _TIMEWORDS
def is_number(t): return bool(re.fullmatch(r"\d+(?:\.\d+)?", t))
def is_proper(t,i): return i>0 and t[:1].isupper() and t.isalpha()

def assign_bio(words, salient, mode=MODE):
    n=len(words); tags=["O"]*n; runs=[]; i=0
    while i<n:
        if salient[i]:
            j=i
            while j+1<n and salient[j+1]: j+=1
            runs.append((i,j)); i=j+1
        else: i+=1
    if mode=="ae_only":
        for s,e in runs:
            tags[s]="B-AE"
            for k in range(s+1,e+1): tags[k]="I-AE"
        return tags
    for s,e in runs:
        tags[e]="B-AE"; started=False
        for k in range(s,e):
            tags[k]="I-DESC" if started else "B-DESC"; started=True
    for idx,tok in enumerate(words):
        if tags[idx]!="O": continue
        if is_time(tok): tags[idx]="B-TIME"
        elif is_number(tok):
            tags[idx]="B-AFF"; k=idx+1
            if k<n and words[k].isalpha() and tags[k]=="O": tags[k]="I-AFF"
        elif is_proper(tok,idx): tags[idx]="B-LOC"
    return tags

## 8 · Run one model

In [10]:
def run_model(text, model, tok, adv_idx, use_masking, gate, mode=MODE):
    words = split_words(text)
    if not words:
        return dict(words=[],tags=[],attn_z=[],delta_p=[],p_adverse=0.0,is_adverse=False,use_masking=use_masking)
    enc, wid = encode_words(words, tok); n=len(words)
    p, out = classify(enc, model, adv_idx)
    attn_z = attention_saliency(out, wid, n)
    delta = masking_saliency(enc, wid, n, p, model, adv_idx, tok) if use_masking else np.zeros(n)
    is_adv = (p >= ADVERSE_GATE) if gate else True
    tags = assign_bio(words, salient_words(attn_z, delta, use_masking), mode) if is_adv else ["O"]*n
    return dict(words=words, tags=tags, attn_z=attn_z.tolist(), delta_p=delta.tolist(),
                p_adverse=p, is_adverse=is_adv, use_masking=use_masking)

def compare(text, mode=MODE):
    ft = run_model(text, ft_model, ft_tok, ADV_IDX, use_masking=True,  gate=True,  mode=mode)
    bl = run_model(text, bl_model, bl_tok, ADV_IDX, use_masking=False, gate=False, mode=mode)
    return ft, bl

## 9 · Quick test — attention focus side by side
The key number is **attn_z**: where does each model peak?

In [11]:
_demo=("A massive 7.8 magnitude earthquake struck Nepal on Saturday, "
       "triggering a deadly avalanche that killed 18 climbers near Everest.")
ft,bl=compare(_demo)
print(f"{'Token':<14}{'FT tag':<9}{'FT z':>7}   {'BL tag':<9}{'BL z':>7}")
print("-"*54)
for w,ft_t,fz,bl_t,bz in zip(ft['words'],ft['tags'],ft['attn_z'],bl['tags'],bl['attn_z']):
    print(f"{w:<14}{ft_t:<9}{fz:>7.2f}   {bl_t:<9}{bz:>7.2f}")
print("-"*54)
i_ft=int(np.argmax(ft['attn_z'])); i_bl=int(np.argmax(bl['attn_z']))
print(f"Fine-tuned peaks on : '{ft['words'][i_ft]}' (z={ft['attn_z'][i_ft]:.2f})")
print(f"Baseline   peaks on : '{bl['words'][i_bl]}' (z={bl['attn_z'][i_bl]:.2f})")

Token         FT tag      FT z   BL tag      BL z
------------------------------------------------------
A             O           0.21   O          -0.17
massive       O           0.82   O          -0.45
7.8           B-AFF      -0.04   B-AFF      -0.68
magnitude     I-AFF       0.30   I-AFF      -0.29
earthquake    B-AE        2.10   O          -0.58
struck        O           0.06   O          -0.41
Nepal         B-LOC       0.85   B-LOC       0.16
on            O          -0.54   O           0.91
Saturday      B-TIME      0.49   B-DESC      1.76
,             O          -1.05   B-AE        2.31
triggering    O          -0.61   O          -0.62
a             O          -0.32   O          -0.50
deadly        O          -0.09   O          -0.57
avalanche     O          -0.06   O          -0.64
that          O          -0.82   O          -0.68
killed        O          -0.63   O          -0.59
18            B-AFF      -1.06   B-AFF      -0.66
climbers      I-AFF      -0.80   I-AFF      -

## 10 · 🔴 Live side-by-side tagger
Type a sentence → see what each model focuses on.

In [12]:
TAG_COLORS={"B-AE":"#e63946","I-AE":"#e63946","B-DESC":"#f4a261","I-DESC":"#f4a261",
            "B-LOC":"#2a9d8f","I-LOC":"#2a9d8f","B-TIME":"#457b9d","I-TIME":"#457b9d",
            "B-AFF":"#9d4edd","I-AFF":"#9d4edd","O":"#cfd8dc"}

def _chips(res):
    out=['<div style="font-family:system-ui;line-height:2.2">']
    for w,t in zip(res["words"],res["tags"]):
        c=TAG_COLORS.get(t,"#cfd8dc"); fg="#fff" if t!="O" else "#607d8b"
        lab="" if t=="O" else f'<sub style="font-size:9px;opacity:.9"> {t}</sub>'
        out.append(f'<span style="background:{c};color:{fg};padding:3px 8px;margin:2px;border-radius:6px;display:inline-block">{w}{lab}</span>')
    out.append("</div>"); return "".join(out)

def _panel(res, title, is_baseline):
    if is_baseline:
        banner=('<div style="background:#fff3cd;border:1px solid #ffe69c;color:#664d03;'
                'padding:6px 10px;border-radius:6px;font-size:12px;margin-bottom:8px">'
                '⚠ random classification head — P(adverse) is meaningless, tags are <b>attention-only</b></div>')
        sub=""
    else:
        adv=res["is_adverse"]; c="#e63946" if adv else "#2a9d8f"
        banner=(f'<div style="font-weight:700;color:{c};margin-bottom:4px">'
                f'{"ADVERSE EVENT" if adv else "NOT adverse"} &nbsp;<span style="font-weight:400;color:#555">P={res["p_adverse"]:.3f}</span></div>')
        sub=""
    top=int(np.argmax(res["attn_z"])) if res["words"] else -1
    rows=""
    for k,(w,t,z) in enumerate(zip(res["words"],res["tags"],res["attn_z"])):
        hl="background:#fff0f0;font-weight:700" if k==top else ""
        rows+=f"<tr style='{hl}'><td style='padding:1px 8px'>{w}</td><td style='padding:1px 8px'>{t}</td><td style='padding:1px 8px;text-align:right'>{z:.2f}</td></tr>"
    table=(f"<table style='font-family:ui-monospace,monospace;font-size:11px;border-collapse:collapse'>"
           f"<tr style='border-bottom:1px solid #ccc'><th style='padding:1px 8px;text-align:left'>tok</th>"
           f"<th style='padding:1px 8px;text-align:left'>tag</th><th style='padding:1px 8px'>attn_z</th></tr>{rows}</table>")
    return (f'<div style="flex:1;min-width:300px;border:1px solid #e0e0e0;border-radius:8px;padding:12px;margin:6px">'
            f'<div style="font-weight:700;font-size:14px;margin-bottom:6px">{title}</div>{banner}{_chips(res)}<div style="margin-top:8px">{table}</div></div>')

def render(ft,bl):
    return f'<div style="display:flex;flex-wrap:wrap">{_panel(ft,"🟢 Fine-tuned (my_bert_model2)",False)}{_panel(bl,"⚪ Baseline (bert-base-uncased)",True)}</div>'

box=widgets.Textarea(value=_demo,layout=widgets.Layout(width="80%",height="60px"))
mode_dd=widgets.Dropdown(options=[("fine","fine"),("ae_only","ae_only")],value=MODE,description="Mode:")
btn=widgets.Button(description="Compare",button_style="primary",icon="play")
out_w=widgets.Output()

def _run(_=None):
    with out_w:
        clear_output()
        txt=box.value.strip()
        if not txt: print("Enter a sentence."); return
        ft,bl=compare(txt,mode=mode_dd.value)
        display(HTML(render(ft,bl)))

btn.on_click(_run)
display(widgets.VBox([box,widgets.HBox([mode_dd,btn]),out_w]))
_run()